# EDA: Injuries


## Purpose
Injury data availability by era. Impact of key player injuries on upset rates. Validate role-specific injury features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg


## Injury Data Availability by Era

In [ ]:
injury_path = cfg.project_root / "data/processed/injury_adjusted.parquet"
if injury_path.exists():
    df = pd.read_parquet(injury_path)
    print(df["has_injury_data"].value_counts())
    print(df.groupby("season")["has_injury_data"].mean())
else:
    print("Run make process first")


## Injury Impact on Upset Rates

In [ ]:
series_path = cfg.project_root / "data/processed/series_dataset.parquet"
sf = pd.read_parquet(series_path)

Path("../reports/figures").mkdir(parents=True, exist_ok=True)

has_data = sf["higher_has_injury_data"].sum() if "higher_has_injury_data" in sf.columns else 0
print(f"Series with injury data: {int(has_data)} / {len(sf)}")

if has_data == 0:
    print("\nNo structured injury data collected (injury_reports not fetched).")
    print("Injury features default to 'available' — model treats all players as healthy.")
    print("\nImpact on modeling:")
    print("  - has_injury_data=0 for all rows; model learns to ignore injury features")
    print("  - Series outcomes still predicted via team stats + player VORP/BPM")
    print("  - Future improvement: wire up injury report scraper to populate these fields")

    # Show the injury feature columns and their default values in the series dataset
    inj_cols = [c for c in sf.columns if "injured" in c.lower() or "lost_top" in c.lower() 
                or "available" in c.lower() or "injury" in c.lower()]
    print(f"\nInjury columns in series dataset ({len(inj_cols)}):")
    if inj_cols:
        print(sf[inj_cols].describe().round(3).to_string())
else:
    # If injury data exists, compute upset rates by injury status
    upset_healthy = sf[(sf["higher_Star_injured"] == 0) & (sf["higher_seed_wins"] == 0)].shape[0]
    total_healthy = sf[sf["higher_Star_injured"] == 0].shape[0]
    upset_injured = sf[(sf["higher_Star_injured"] == 1) & (sf["higher_seed_wins"] == 0)].shape[0]
    total_injured = sf[sf["higher_Star_injured"] == 1].shape[0]
    print(f"Upset rate — healthy higher seed:  {upset_healthy/total_healthy:.1%}")
    print(f"Upset rate — injured higher seed:   {upset_injured/total_injured:.1%}")


## Role-Specific Injury Validation

In [ ]:
# Validate known injury events in our data
# Since has_injury_data=0 everywhere, validate the pipeline structure instead

inj_df = pd.read_parquet(cfg.project_root / "data/processed/injury_adjusted.parquet")

print("=== Injury Feature Defaults (no live injury data) ===")
print(inj_df[inj_df["season"] == 2026][["Team_abbrev", "has_injury_data",
    "Roster_VORP_available_pct", "Star_injured", "Injured_player_count"]].to_string(index=False))

print("\n=== What correct data would look like ===")
print("2019 Warriors (Kevin Durant torn Achilles, Klay Thompson torn ACL):")
print("  Expected: Star_injured=1, Second_star_injured=1, Roster_VORP_available_pct ≈ 0.5-0.6")
print("2014 Spurs (fully healthy):") 
print("  Expected: Star_injured=0, Roster_VORP_available_pct ≈ 1.0")

print("\n=== Roster_VORP_available_pct distribution (all seasons) ===")
print(inj_df["Roster_VORP_available_pct"].describe().round(3))
print("\nAll values are 1.0 → confirms injury features are placeholder defaults")
